In [59]:
import requests
import json

# Your Qualtrics API token and Data Center ID
API_TOKEN = "QUahqaqAx3GWF0pswKjbYYR37cil6xd0vN2zpB85" # your API ID
DATA_CENTER = 'uofsc.sjc1' # "<Organization ID>.<Datacenter ID>"

# Base URL for API requests
BASE_URL = f"https://{DATA_CENTER}.qualtrics.com/API/v3" #f"https://{DATA_CENTER}.qualtrics.com/API/v3"

# Headers for API authentication
# HEADERS = {
#     "Content-Type": "application/json",
#     "X-API-TOKEN": API_TOKEN
# }
HEADERS = {
    "x-api-token": API_TOKEN,
    "content-type": "application/json",
    "Accept": "application/json"
}

# Step 1: Create a new survey
def create_survey(survey_name):
    url = f"{BASE_URL}/survey-definitions"
    data = {"SurveyName": survey_name,
            "Language": "EN",
            "ProjectCategory": "CORE"}

    response = requests.post(url, json=data, headers=HEADERS)
    # print(response.text)
    survey_id = json.loads(response.text)["result"]["SurveyID"]
    print(f"Survey Created: {survey_id}")
    return survey_id


In [58]:
survey_name = 'test_APIproject'
surveyID = create_survey(survey_name)

{"meta":{"httpStatus":"200 - OK","requestId":"846c41fd-d17a-4d8b-bd92-52a4083c660f"},"result":{"SurveyID":"SV_eRHmYL42e3ZtMb4","DefaultBlockID":"BL_bvVRmGgJ7f08sN8"}}
Survey Created: SV_eRHmYL42e3ZtMb4


In [64]:
def add_EmbeddedData(surveyID,data):
    url = f"{BASE_URL}/surveys/{surveyID}/embeddeddatafields"

    response = requests.post(url, json=data, headers=HEADERS)
    
    # print(f"Survey Created: {survey_id}")
    return response

In [ ]:
# data = {"key": "id","type":"string"}
data = {
  "embeddedDataFields": [
    {"key": "id","type": "text"}
    #   "value": "string"
  ]
}
response = add_EmbeddedData(surveyID,data)
# response.text

'{"meta":{"httpStatus":"200 - OK","requestId":"bbf9955b-a472-4e0e-abf8-cf679f0573cc"}}'

'{"meta":{"requestId":"affb0b66-a592-4b06-90e8-d6a191871dc8","httpStatus":"404 - Not Found","error":{"errorMessage":"The requested resource does not exist."}}}'

In [55]:
import requests

# Setting user Parameters
apiToken = "API_TOKEN" # Replace with Account API Token
dataCenter = "ca1"

surveyID = "SV_096ecQyLMysaaEZ"

baseUrl = "https://{0}.qualtrics.com/API/v3/survey-definitions/{1}/questions".format(
    dataCenter, surveyId)

questionDef = {
  "QuestionText": "Color?",
  "DataExportTag": "Q1",
  "QuestionType": "MC",
  "Selector": "SAVR",
  "SubSelector": "TX",
  "Configuration": {
    "QuestionDescriptionOption": "UseText"
  },
  "QuestionDescription": "Color?",
  "Choices": {
    "1": {
      "Display": "red"
    },
    "2": {
      "Display": "blue"
    }
  },
  "ChoiceOrder": [
    "1",
    "2"
  ],
  "Validation": {
    "Settings": {
      "ForceResponse": "OFF",
      "ForceResponseType": "ON",
      "Type": "None"
    }
  },
  "Language": [],
  "NextChoiceId": 3,
  "NextAnswerId": 1,
  "QuestionID": "QID1",
  "QuestionText_Unsafe": "Color?"
}

headers = {
   'accept': "application/json",
   'content-type': "application/json",
   "x-api-token": apiToken,
}

response = requests.post(baseUrl, json=questionDef, headers=headers)

print(response.text)

<Response [200]>

In [ ]:
import os
import requests

# Setting user Parameters
apiToken = "QUahqaqAx3GWF0pswKjbYYR37cil6xd0vN2zpB85" # your API ID
dataCenter = 'uofsc.sjc1' # "<Organization ID>.<Datacenter ID>"

baseUrl = f"https://{dataCenter}.qualtrics.com/API/v3/survey-definitions"
headers = {
    "x-api-token": apiToken,
    "content-type": "application/json",
    "Accept": "application/json"
}


survey_name = 'Test_APIproject'
data = {"SurveyName": survey_name,"Language": "EN","ProjectCategory": "CORE"}

response = requests.post(baseUrl, json=data, headers=headers)
print(response.text)

{"meta":{"httpStatus":"200 - OK","requestId":"7c65a519-8561-49f1-ac21-cdf30bc6cee1"},"result":{"SurveyID":"SV_bpIijb02HKpjZ6m","DefaultBlockID":"BL_9oCrnUHaj3Ch0Gi"}}


In [20]:
data_dict = json.loads(response.text)
data_dict['result']['SurveyID']

'SV_bpIijb02HKpjZ6m'

In [ ]:

# Step 2: Add a question with embedded data and branching
def add_questions(survey_id):
    url = f"{BASE_URL}/survey-definitions/{survey_id}"
    
    # Define question structure
    question_data = {
        "SurveyElements": [
            {
                "SurveyID": survey_id,
                "Element": "SQ",
                "PrimaryAttribute": "Q1",
                "SecondaryAttribute": "Do you like Python?",
                "Payload": {
                    "QuestionText": "Do you like Python?",
                    "DataExportTag": "Q1",
                    "QuestionType": "MC",
                    "Selector": "SAVR",
                    "Configuration": {"QuestionDescriptionOption": "UseText"},
                    "Choices": {"1": {"Display": "Yes"}, "2": {"Display": "No"}},
                    "QuestionID": "Q1",
                    "Validation": {"Settings": {"ForceResponse": "true"}}
                }
            }
        ]
    }

    response = requests.put(url, headers=HEADERS, json=question_data)
    print(f"Questions Added: {response.status_code}")

# Step 3: Add If-Then Branching
def add_branching(survey_id):
    url = f"{BASE_URL}/survey-definitions/{survey_id}"
    
    branch_data = {
        "SurveyElements": [
            {
                "SurveyID": survey_id,
                "Element": "BL",
                "PrimaryAttribute": "Branch1",
                "Payload": {
                    "Type": "BooleanExpression",
                    "Description": "Branching based on Q1",
                    "Logic": {
                        "0": {
                            "Type": "Expression",
                            "QuestionID": "Q1",
                            "Operator": "Selected",
                            "LeftOperand": "1"
                        }
                    },
                    "Consequence": {
                        "Display": ["Q2"]
                    }
                }
            }
        ]
    }

    response = requests.put(url, headers=HEADERS, json=branch_data)
    print(f"Branching Logic Added: {response.status_code}")

# Step 4: Add Web Service Integration
def add_webservice(survey_id):
    url = f"{BASE_URL}/survey-definitions/{survey_id}"

    web_service_data = {
        "SurveyElements": [
            {
                "SurveyID": survey_id,
                "Element": "WS",
                "PrimaryAttribute": "WebService1",
                "Payload": {
                    "Type": "WebService",
                    "URL": "https://your-backend.com/api",
                    "Method": "GET",
                    "Headers": {"Authorization": "Bearer your_token"},
                    "Parameters": {"participant_id": "${e://Field/ParticipantID}"},
                    "EmbeddedDataMapping": {
                        "StudyStatus": "study_status",
                        "PreviousStories": "previous_stories"
                    }
                }
            }
        ]
    }

    response = requests.put(url, headers=HEADERS, json=web_service_data)
    print(f"Web Service Added: {response.status_code}")

# Step 5: Publish Survey
def publish_survey(survey_id):
    url = f"{BASE_URL}/surveys/{survey_id}/versions"
    response = requests.post(url, headers=HEADERS)
    print(f"Survey Published: {response.status_code}")

# Running the full process
survey_id = create_survey()
add_questions(survey_id)
add_branching(survey_id)
add_webservice(survey_id)
publish_survey(survey_id)
